# cAPTure: canonical packet preparation

This CPU-only notebook converts the completed Gate-0 audit Parquet files into the window-independent candidate `capture_packet_v1` schema. It reads the existing files from Drive, processes one scenario at a time, and writes checksum-verified prepared artifacts back to Drive. It does not download source CSVs, assign packets to windows, compute training weights, or train a model.

Run `SMOKE` first. Review the real-data transformations before approving and running `FULL_DEV`.


## 1. Mount Drive and load the project


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_ROOT = Path("/content/capture_prepare_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_prepare.py",
    PROJECT_ROOT / "code/python/tests/test_capture_prepare.py",
    PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("CPU preparation environment is ready.")


Mounted at /content/drive
CPU preparation environment is ready.


## 2. Run synthetic transformation checks


In [2]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_prepare.py", "-v"],
    env=test_environment,
    cwd=PROJECT_ROOT,
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/temporalgnn-nids/code/python/tests', '-p', 'test_capture_prepare.py', '-v'], returncode=0)

## 3. Configure the preparation run


In [7]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from utils.capture_data import load_manifest, selected_scenarios, sha256_file, write_json
from utils.capture_prepare import run_capture_preparation

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
AUDIT_RUN_DIR = (
    DRIVE_ROOT / "runs" / "20260917T170658_227477Z_full_dev"
)
DECISION_AUDIT_PATH = AUDIT_RUN_DIR / "gate0_decision_audit.json"

#MODE = "SMOKE"
MODE = "FULL_DEV"
PREPARATION_SMOKE_REVIEW_PATH = (
    "/content/drive/MyDrive/capture_gate0/prepared_runs/"
    "20260917T233015_342509Z_prepare_smoke/preparation_smoke_review.json"
)
BATCH_SIZE = 100_000

MANIFEST = load_manifest(MANIFEST_PATH)
SCENARIOS = selected_scenarios(MANIFEST, MODE)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_prepare_" + MODE.lower()
DRIVE_RUN_DIR = DRIVE_ROOT / "prepared_runs" / RUN_ID

print(f"Mode: {MODE}")
print(f"Scenarios: {SCENARIOS}")
print(f"Source audit: {AUDIT_RUN_DIR}")
print(f"Output directory: {DRIVE_RUN_DIR}")
print(f"Free local storage: {shutil.disk_usage(LOCAL_ROOT).free / 1024**3:.1f} GiB")


Mode: FULL_DEV
Scenarios: ['train_empty_conn', 'train_qos_mid', 'train_dollar_char', 'train_slash_char', 'train_sub_exf']
Source audit: /content/drive/MyDrive/capture_gate0/runs/20260917T170658_227477Z_full_dev
Output directory: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Free local storage: 85.2 GiB


## 4. Prepare and persist the selected scenarios

The input audit Parquet files remain on Drive. Each output is built in temporary local storage, copied to a fresh Drive run directory, checksum-verified, and then removed locally.


In [8]:
RESULTS = run_capture_preparation(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    audit_run_dir=AUDIT_RUN_DIR,
    decision_audit_path=DECISION_AUDIT_PATH,
    mode=MODE,
    local_root=LOCAL_ROOT,
    drive_run_dir=DRIVE_RUN_DIR,
    smoke_review_path=(
        Path(PREPARATION_SMOKE_REVIEW_PATH)
        if PREPARATION_SMOKE_REVIEW_PATH is not None else None
    ),
    batch_size=BATCH_SIZE,
)
display(pd.DataFrame([
    {"scenario": scenario, **item}
    for scenario, item in RESULTS.items()
]))


Preparing train_empty_conn. Local workspace: /content/capture_prepare_work/train_empty_conn_fa1bihqc
train_empty_conn: prepared 100,000 packets
train_empty_conn: prepared 200,000 packets
train_empty_conn: prepared 300,000 packets
train_empty_conn: prepared 400,000 packets
train_empty_conn: prepared 500,000 packets
train_empty_conn: prepared 600,000 packets
train_empty_conn: prepared 700,000 packets
train_empty_conn: prepared 800,000 packets
train_empty_conn: prepared 900,000 packets
train_empty_conn: prepared 1,000,000 packets
train_empty_conn: prepared 1,100,000 packets
train_empty_conn: prepared 1,175,779 packets
Saved and verified train_empty_conn on Drive. Removed its local copy.
Preparing train_qos_mid. Local workspace: /content/capture_prepare_work/train_qos_mid_krumhk73
train_qos_mid: prepared 100,000 packets
train_qos_mid: prepared 200,000 packets
train_qos_mid: prepared 300,000 packets
train_qos_mid: prepared 400,000 packets
train_qos_mid: prepared 500,000 packets
train_qos_mi

,scenario,status,report
0,train_empty_conn,review_required,/content/drive/MyDrive/capture_gate0/prepared_...
1,train_qos_mid,review_required,/content/drive/MyDrive/capture_gate0/prepared_...
2,train_dollar_char,review_required,/content/drive/MyDrive/capture_gate0/prepared_...
3,train_slash_char,review_required,/content/drive/MyDrive/capture_gate0/prepared_...
4,train_sub_exf,review_required,/content/drive/MyDrive/capture_gate0/prepared_...


## 5. Review the preparation reports


In [9]:
REPORTS = {}
summary_rows = []
role_rows = []
null_rows = []

for scenario in SCENARIOS:
    report_path = DRIVE_RUN_DIR / scenario / "preparation_report.json"
    report = json.loads(report_path.read_text())
    REPORTS[scenario] = report
    summary_rows.append({
        "scenario": scenario,
        **report["counts"],
        "output_gib": report["output_size_bytes"] / 1024**3,
        "features": len(report["feature_columns"]),
        "origin_timestamp_ns": report["scenario_origin_timestamp_ns"],
        "last_timestamp_ns": report["last_packet_timestamp_ns"],
        "duration_seconds": report["duration_seconds"],
    })
    role_rows.append({"scenario": scenario, **report["destination_node_role_counts"]})
    for feature, missing in report["feature_null_counts"].items():
        null_rows.append({
            "scenario": scenario, "feature": feature, "null_values": missing,
            "null_fraction": missing / report["counts"]["packets"],
        })

print("Scenario summary")
display(pd.DataFrame(summary_rows))
print("Destination node roles")
display(pd.DataFrame(role_rows))
print("Feature null fractions")
null_frame = pd.DataFrame(null_rows)
for scenario in SCENARIOS:
    print(scenario)
    display(
        null_frame[null_frame["scenario"] == scenario]
        .drop(columns="scenario")
        .reset_index(drop=True)
    )


Scenario summary


,scenario,packets,normal_packets,attack_packets,output_gib,features,origin_timestamp_ns,last_timestamp_ns,duration_seconds
0,train_empty_conn,1175779,746806,428973,0.061369,42,19792000,46799813163000,46799.793371
1,train_qos_mid,1499717,746806,752911,0.078339,42,19792000,46799813163000,46799.793371
2,train_dollar_char,2882555,1705003,1177552,0.148095,42,9927000,46796778032000,46796.768105
3,train_slash_char,6425516,1705003,4720513,0.315006,42,9927000,46796778032000,46796.768105
4,train_sub_exf,2225807,1705003,520804,0.112849,42,9927000,46796778032000,46796.768105


Destination node roles


,scenario,broadcast,multicast,unicast
0,train_empty_conn,19805,27000,1128974
1,train_qos_mid,20980,26240,1452497
2,train_dollar_char,26334,23599,2832622
3,train_slash_char,53481,22114,6349921
4,train_sub_exf,94429,22280,2109098


Feature null fractions
train_empty_conn


...

,feature,null_values,null_fraction
0,destination_is_broadcast,0,0.000000
1,destination_is_multicast,0,0.000000
2,ethernet_type,0,0.000000
3,frame_length,0,0.000000
4,ipv4_dscp,81195,0.028168
5,ipv4_flag_df,81195,0.028168
6,ipv4_flag_mf,81195,0.028168
7,ipv4_fragment_offset,81195,0.028168
8,ipv4_length,81195,0.028168
9,ipv4_ttl,81195,0.028168


train_slash_char


,feature,null_values,null_fraction
0,destination_is_broadcast,0,0.000000
1,destination_is_multicast,0,0.000000
2,ethernet_type,0,0.000000
3,frame_length,0,0.000000
4,ipv4_dscp,104666,0.016289
5,ipv4_flag_df,104666,0.016289
6,ipv4_flag_mf,104666,0.016289
7,ipv4_fragment_offset,104666,0.016289
8,ipv4_length,104666,0.016289
9,ipv4_ttl,104666,0.016289


train_sub_exf


,feature,null_values,null_fraction
0,destination_is_broadcast,0,0.000000
1,destination_is_multicast,0,0.000000
2,ethernet_type,0,0.000000
3,frame_length,0,0.000000
4,ipv4_dscp,154962,0.069621
5,ipv4_flag_df,154962,0.069621
6,ipv4_flag_mf,154962,0.069621
7,ipv4_fragment_offset,154962,0.069621
8,ipv4_length,154962,0.069621
9,ipv4_ttl,154962,0.069621


## 6. Record the preparation SMOKE review

Set `APPROVE_PREPARATION_SMOKE` to `True` only after reviewing Section 5. The resulting JSON authorizes `FULL_DEV` for this exact manifest, packet schema, and pair of SMOKE reports.


In [6]:
APPROVE_PREPARATION_SMOKE = True
PREPARATION_REVIEW_NOTES = (
    "Canonical packet counts, timestamp ranges, parsed features, node roles, "
    "and null fractions were reviewed without blockers."
)

if MODE != "SMOKE":
    print("This section applies only to a SMOKE preparation run.")
elif APPROVE_PREPARATION_SMOKE:
    review = {
        "approved": True,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "packet_schema_sha256": sha256_file(PACKET_SCHEMA_PATH),
        "review_notes": PREPARATION_REVIEW_NOTES,
        "reports": {},
    }
    for scenario in SCENARIOS:
        report_path = DRIVE_RUN_DIR / scenario / "preparation_report.json"
        review["reports"][scenario] = {
            "path": str(report_path),
            "sha256": sha256_file(report_path),
        }
    review_path = DRIVE_RUN_DIR / "preparation_smoke_review.json"
    write_json(review_path, review)
    print(f"Preparation SMOKE approved: {review_path}")
else:
    print("Preparation SMOKE review remains pending. No approval was recorded.")


Preparation SMOKE approved: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T233015_342509Z_prepare_smoke/preparation_smoke_review.json
